# RVCBench FishSpeech Quickstart

This notebook shows a minimal Colab-style workflow for running `RVCBench` with a lightweight zero-shot cloning model: **FishSpeech S1-mini**.

What it does:
- clones this repository,
- installs the small set of dependencies needed for the FishSpeech path,
- downloads a tiny **VCTK** subset from the public `Nanboy/RVCBench` dataset,
- clones the Fish-Speech inference repo,
- downloads the gated `fishaudio/s1-mini` checkpoint,
- runs `run_vc.py` on a few samples,
- previews the generated audio and metrics.

Notes:
- A Colab GPU runtime is strongly recommended.
- `fishaudio/s1-mini` is gated on Hugging Face, so you must accept its model terms first and use a token with access.
- This notebook uses a tiny subset and `vc.max_samples` to illustrate the workflow, not to reproduce full benchmark numbers.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/rjin02/RVCBench.git"
WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "RVCBench"

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Repository:", REPO_DIR)


In [ ]:
%%bash
set -euxo pipefail

python -m pip install --upgrade pip
python -m pip install -q \
  hydra-core \
  omegaconf \
  pandas \
  pyarrow \
  jiwer \
  openai-whisper \
  pymcd \
  librosa \
  soundfile \
  matplotlib \
  huggingface_hub


## Authenticate for gated FishSpeech weights

Before running the next cell:
1. Open the model page and accept access terms: `https://huggingface.co/fishaudio/s1-mini`
2. Create or reuse a Hugging Face token that can read gated models.


In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token with access to fishaudio/s1-mini: ")

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF login configured.")


In [ ]:
from huggingface_hub import snapshot_download

DATA_ROOT = REPO_DIR / "data"
DATASET_ROOT = DATA_ROOT / "VCTK"
SPEAKER_ID = "p225"

snapshot_download(
    repo_id="Nanboy/RVCBench",
    repo_type="dataset",
    local_dir=str(DATA_ROOT),
    allow_patterns=[
        "VCTK/metadata.parquet",
        f"VCTK/audios/{SPEAKER_ID}/*.wav",
        f"VCTK/audios/{SPEAKER_ID}/*.bert.pt",
    ],
)

print("Dataset root:", DATASET_ROOT)
print("Example files:")
for path in sorted((DATASET_ROOT / "audios" / SPEAKER_ID).glob("*.wav"))[:5]:
    print(" -", path.name)


In [ ]:
%%bash
set -euxo pipefail

mkdir -p checkpoints
if [ ! -d checkpoints/fish_speech/.git ]; then
  git clone https://github.com/fishaudio/fish-speech.git checkpoints/fish_speech
fi

python -m pip install -q -e checkpoints/fish_speech


In [ ]:
from huggingface_hub import snapshot_download

FISHSPEECH_CKPT = REPO_DIR / "checkpoints" / "fish_speech" / "openaudio-s1-mini"

snapshot_download(
    repo_id="fishaudio/s1-mini",
    local_dir=str(FISHSPEECH_CKPT),
    token=os.environ["HF_TOKEN"],
)

print("Checkpoint root:", FISHSPEECH_CKPT)
print("codec exists:", (FISHSPEECH_CKPT / "codec.pth").exists())


In [ ]:
import torch

os.environ["DATA_LOADER_WORKERS"] = "0"
os.environ["AUDIOBENCH_FORCE_OFFLINE"] = "0"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES = 3
os.environ["DEVICE"] = DEVICE
os.environ["MAX_SAMPLES"] = str(MAX_SAMPLES)
os.environ["SPEAKER_ID"] = SPEAKER_ID

print("DEVICE =", DEVICE)
print("MAX_SAMPLES =", MAX_SAMPLES)


In [ ]:
%%bash
set -euxo pipefail

python run_vc.py \
  --config-name ots_vc/clean/vctk/fishspeech_ots \
  device=${DEVICE} \
  dataset.root_path=./data/VCTK \
  dataset.speaker_id=${SPEAKER_ID} \
  batch_size=1 \
  adversary.max_samples=${MAX_SAMPLES} \
  adversary.code_path=./checkpoints/fish_speech \
  adversary.llama_checkpoint_path=./checkpoints/fish_speech/openaudio-s1-mini \
  adversary.decoder_checkpoint_path=./checkpoints/fish_speech/openaudio-s1-mini/codec.pth


In [ ]:
import json
from IPython.display import Audio, display

result_root = max((REPO_DIR / "results" / "fishspeech_ots_on_vctk").glob("*"), key=lambda p: p.stat().st_mtime)
metrics_path = result_root / "metrics.json"
generated_dir = result_root / "generated_audio"

print("Latest result:", result_root)
print("Metrics file:", metrics_path)

metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics.get("generation_evaluation", {}), indent=2)[:4000])

generated_files = sorted(generated_dir.rglob("*.wav"))
print(f"Generated {len(generated_files)} wav files")
display(Audio(str(generated_files[0])))


## What to change next

Useful overrides once the quickstart works:
- Increase sample count: `vc.max_samples=10`
- Change dataset speaker: `dataset.speaker_id=p226`
- Run another model by swapping the config name, for example `ots_vc/clean/vctk/ozspeech_ots`
- Point `dataset.root_path` to a full local benchmark snapshot instead of the tiny notebook subset
- Move to protected-audio experiments with `run_vc_protect.py`
